# Классификация мусора по фото

Курсовая работа по курсу Computer Vision.

**Задача:** классификация фотографий отходов на 6 классов — картон, стекло, металл, бумага, пластик, прочее.

**Модель:** MobileNetV2 с предобученными весами ImageNet ([статья](https://arxiv.org/abs/1801.04381)).

**Данные:** [Garbage Classification на Kaggle](https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification).

## 1. Импорты и настройки

In [ ]:
import os
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
CLASSES_RU = {
    'cardboard': 'картон',
    'glass': 'стекло',
    'metal': 'металл',
    'paper': 'бумага',
    'plastic': 'пластик',
    'trash': 'прочее',
}

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Загрузка и разведка данных

Архив `garbage_classification.zip` с Kaggle нужно заранее положить в корень Google Drive.

В архиве датасет лежит дважды: в папках `Garbage classification` и `garbage classification` (отличаются только регистром, файлы одинаковые). Поэтому распаковываем только одну копию.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/garbage_classification.zip'

!unzip -q -n "{ZIP_PATH}" "Garbage classification/*" -d /content/raw

DATA_DIR = '/content/raw/Garbage classification/Garbage classification'
print(sorted(os.listdir(DATA_DIR)))

### Количество изображений по классам

In [ ]:
counts = [len(os.listdir(os.path.join(DATA_DIR, cls))) for cls in CLASSES]

df_counts = pd.DataFrame({
    'класс': [CLASSES_RU[cls] for cls in CLASSES],
    'количество': counts,
})
df_counts['доля, %'] = (df_counts['количество'] / df_counts['количество'].sum() * 100).round(1)

print('Всего изображений:', df_counts['количество'].sum())
df_counts

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(df_counts['класс'], df_counts['количество'], color='steelblue')
for i, value in enumerate(df_counts['количество']):
    plt.text(i, value + 5, str(value), ha='center')
plt.title('Количество изображений по классам')
plt.ylabel('Количество')
plt.tight_layout()
plt.show()

### Примеры изображений

In [ ]:
n_examples = 4
fig, axes = plt.subplots(len(CLASSES), n_examples, figsize=(n_examples * 3, len(CLASSES) * 2.4))

for row, cls in enumerate(CLASSES):
    files = random.sample(os.listdir(os.path.join(DATA_DIR, cls)), n_examples)
    for col, fname in enumerate(files):
        img = Image.open(os.path.join(DATA_DIR, cls, fname))
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
    axes[row, 0].set_title(CLASSES_RU[cls], loc='left')

plt.tight_layout()
plt.show()

### Размеры и цветовой режим изображений

In [ ]:
rows = []
for cls in CLASSES:
    for fname in os.listdir(os.path.join(DATA_DIR, cls)):
        with Image.open(os.path.join(DATA_DIR, cls, fname)) as img:
            rows.append({'ширина': img.width, 'высота': img.height, 'режим': img.mode})

pd.DataFrame(rows).value_counts()

### Выводы по данным

- В датасете 2527 изображений, 6 классов.
- Классы несбалансированы: больше всего бумаги (594, 23.5%), меньше всего класса «прочее» (137, 5.4%). Это нужно учесть: делить данные со стратификацией и смотреть не только на accuracy, но и на метрики по классам.
- Все изображения одного размера 512×384 в RGB. В кадре один объект, в основном на однотонном светлом фоне, часть фото сделана крупным планом.
- Некоторые классы визуально похожи: например, прозрачные бутылки из стекла и пластика. Скорее всего, модель будет их путать.
- Данных мало (около 400 фото на класс), поэтому обучать большую сеть с нуля будет тяжело. Это аргумент в пользу transfer learning с предобученной на ImageNet моделью.

## 3. Препроцессинг и разделение на train/val/test

Делим данные на три части в пропорции 70/15/15:
- **train** — на ней обучаем модель;
- **val** — по ней следим за обучением и выбираем лучшую эпоху;
- **test** — откладываем до самого конца и используем только для итоговой оценки.

Разбиение делаем **со стратификацией**: доля каждого класса во всех трёх частях остаётся такой же, как во всём датасете. Без этого в маленький test могло бы попасть слишком мало фото класса «прочее».

In [ ]:
filepaths = []
labels = []
for label, cls in enumerate(CLASSES):
    for fname in sorted(os.listdir(os.path.join(DATA_DIR, cls))):
        filepaths.append(os.path.join(DATA_DIR, cls, fname))
        labels.append(label)

filepaths = np.array(filepaths)
labels = np.array(labels)

# сначала отделяем 30% под val+test, потом делим их пополам
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    filepaths, labels, test_size=0.3, stratify=labels, random_state=SEED)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=SEED)

print('train:', len(train_paths), ' val:', len(val_paths), ' test:', len(test_paths))

In [ ]:
df_split = pd.DataFrame({
    'train': np.bincount(train_labels),
    'val': np.bincount(val_labels),
    'test': np.bincount(test_labels),
}, index=[CLASSES_RU[cls] for cls in CLASSES])
df_split.loc['всего'] = df_split.sum()
df_split

### Загрузка изображений через tf.data

Все картинки приводим к размеру **224×224** — это стандартный размер входа MobileNetV2. Исходные фото 4:3, так что пропорции немного искажаются, но для классификации это не критично.

Пиксели оставляем в диапазоне 0–255. Нормализацию каждая модель делает сама в первом слое: простая CNN делит на 255, а MobileNetV2 использует свою функцию `preprocess_input`.

`cache()` сохраняет уже декодированные картинки в памяти, чтобы не читать JPEG с диска на каждой эпохе, а `prefetch()` готовит следующий батч, пока GPU считает текущий.

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img, label


def make_dataset(paths, labels, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE).cache()
    if shuffle:
        ds = ds.shuffle(len(paths), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)


train_ds = make_dataset(train_paths, train_labels, shuffle=True)
val_ds = make_dataset(val_paths, val_labels)
test_ds = make_dataset(test_paths, test_labels)

images, batch_labels = next(iter(train_ds))
print('Батч:', images.shape, images.dtype)
print('Диапазон пикселей:', float(tf.reduce_min(images)), '-', float(tf.reduce_max(images)))

### Аугментация

Данных мало, поэтому на обучении будем случайно менять картинки: отражать, поворачивать, приближать, менять контраст. Так модель каждый раз видит немного другое изображение и меньше переобучается.

Отражаем и по горизонтали, и по вертикали: многие объекты сняты сверху, и у них нет «верха» и «низа».

Аугментация сделана слоями Keras. Мы добавим её внутрь моделей: такие слои работают только во время обучения, а на val и test автоматически отключаются.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
], name='augmentation')

img = images[0]
plt.figure(figsize=(12, 5))
for i in range(8):
    augmented = data_augmentation(img[tf.newaxis], training=True)[0]
    plt.subplot(2, 4, i + 1)
    plt.imshow(np.clip(augmented.numpy(), 0, 255).astype('uint8'))
    plt.axis('off')
plt.suptitle('Аугментация одного изображения (класс: ' + CLASSES_RU[CLASSES[int(batch_labels[0])]] + ')')
plt.tight_layout()
plt.show()

### Веса классов

Чтобы модель не игнорировала редкий класс «прочее», при обучении будем передавать веса классов: ошибка на редком классе штрафуется сильнее. Веса считаются по train как `всего / (число_классов × размер_класса)`.

In [ ]:
weights = compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=train_labels)
class_weight = dict(enumerate(weights))

pd.Series(weights.round(2), index=[CLASSES_RU[cls] for cls in CLASSES], name='вес класса')

## 4. Бейзлайн: простая CNN

Сначала обучим небольшую свёрточную сеть с нуля, без предобученных весов. Это точка отсчёта: потом сравним с ней MobileNetV2 и увидим, сколько даёт transfer learning.

Архитектура:
- аугментация (работает только при обучении) и нормализация пикселей в диапазон 0–1;
- 4 свёрточных блока: свёртка 3×3 + ReLU + MaxPooling, число фильтров 32 → 64 → 128 → 128. Каждый MaxPooling уменьшает картинку в 2 раза: 224 → 112 → 56 → 28 → 14;
- GlobalAveragePooling вместо Flatten: усредняет каждую карту признаков в одно число. Получается 128 признаков вместо 14·14·128 = 25088, параметров намного меньше, и сеть слабее переобучается;
- Dropout 0.3 и выходной слой на 6 классов с softmax.

In [ ]:
baseline_model = tf.keras.Sequential([
    tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    data_augmentation,
    tf.keras.layers.Rescaling(1. / 255),

    tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(CLASSES), activation='softmax'),
], name='baseline_cnn')

baseline_model.summary()

### Обучение

- Оптимизатор Adam с learning rate 0.001.
- Функция потерь `sparse_categorical_crossentropy`: метки у нас числами 0–5, а не one-hot векторами.
- Передаём веса классов, посчитанные выше.
- `EarlyStopping` останавливает обучение, если loss на валидации не улучшается 7 эпох подряд, и возвращает веса с лучшей эпохи.

In [ ]:
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=7, restore_best_weights=True)

baseline_history = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    class_weight=class_weight,
    callbacks=[early_stop],
)

In [ ]:
def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history['accuracy'], label='train')
    axes[0].plot(history.history['val_accuracy'], label='val')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Эпоха')
    axes[0].legend()

    axes[1].plot(history.history['loss'], label='train')
    axes[1].plot(history.history['val_loss'], label='val')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Эпоха')
    axes[1].legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_history(baseline_history, 'Простая CNN')

In [ ]:
val_loss, val_acc = baseline_model.evaluate(val_ds, verbose=0)
print(f'Простая CNN, точность на валидации: {val_acc:.3f}')

## 5. MobileNetV2 с transfer learning

## 6. Fine-tuning

## 7. Оценка результатов

## 8. Демо

## 9. Выводы